In [ ]:
from tmkb.configuration import load_config, ConfigSchema
from tmkb.kb_modules import (
    OctaveBuilder,
    MxKeyBuilder,
    MxKeyType,
    CrontrollerCaseBuilder,
    EndCapBuilder,
)
from tmkb.kb_modules.protocols import HasKeys
from tmkb.generate import _get_partial_keys_cnt
from tmkb.primitives.base_builders import RelativeCoords, XPos, YPos, ZPos

from pathlib import Path
from jupyterscad import view

In [ ]:
conf = load_config(Path("../configs/default.conf.yaml"))
conf

In [ ]:
def key_to_preview(builder: MxKeyBuilder):
    builder.move([conf.mount_u / 2, conf.mount_u / 2, conf.mount_u / 2])
    return builder.build(rotation=180)


def build_with_keys(builder: HasKeys, conf: ConfigSchema):
    model = builder.build(printing_orientation=False)
    for type_ in MxKeyType:
        for x, y, z in builder.get_total_key_positions(type_):
            key: MxKeyBuilder = MxKeyBuilder.mx_key_factory(type_, conf)
            key.move([x, y, z])
            model += key_to_preview(key)
    return model

In [ ]:
partial_octave_num_of_keys = _get_partial_keys_cnt(conf.key_count)
keys_per_octave = [7] * (conf.key_count // 12) + [partial_octave_num_of_keys[0]]

octaves = [OctaveBuilder(n, conf) for n in keys_per_octave]

case_ = CrontrollerCaseBuilder(octaves[0], conf)
cap = EndCapBuilder(octaves[0], conf)

model = cap.build(printing_orientation=False)
prev = cap
for el in reversed([case_] + octaves):
    el.move_rel(
        prev,
        anchor=RelativeCoords(xpos=XPos.LEFT),
        alignment=RelativeCoords(ypos=YPos.CENTER),
    )
    model += build_with_keys(el, conf)
    prev = el


view(model)